# Lab 09: Deployment & Model Serving

## Overview

In this lab you will deploy a registered Unity Catalog model as a **Databricks Model Serving endpoint**, test it via the REST API, configure an **A/B traffic split** across two model versions, and run **batch inference** using the `ai_query()` SQL function.

| Topic | Detail |
|---|---|
| **Exam Domain** | Assembling and Deploying Apps (22%) |
| **Key Skills** | Serving endpoints · Scale-to-zero · A/B testing · `ai_query()` · Provisioned vs pay-per-token |

### What you will do
1. Deploy a Unity Catalog model to a Model Serving endpoint using the Databricks SDK.
2. Test the endpoint with a REST query.
3. Configure A/B testing with a 50/50 traffic split between two model versions.
4. Run batch inference from SQL using `ai_query()`.
5. Compare **Provisioned Throughput** vs **Pay-per-Token** (serverless) pricing models.

## Architecture Diagram

![Architecture Diagram](../assets/diagrams/lab-09-deployment-model-serving.png)

In [ ]:
# Prerequisites — run once per session
%pip install databricks-sdk --quiet
dbutils.library.restartPython()

In [ ]:
# Configuration
CATALOG = "genai_lab_guide"
SCHEMA = "default"
ENDPOINT_NAME  = "genai-lab-agent-endpoint"
MODEL_NAME     = f"{CATALOG}.{SCHEMA}.arxiv_research_agent"   # Fully-qualified UC model name
MODEL_VERSION  = 1

print(f"Catalog  : {CATALOG}")
print(f"Schema   : {SCHEMA}")
print(f"Endpoint : {ENDPOINT_NAME}")
print(f"Model    : {MODEL_NAME}  (version {MODEL_VERSION})")

## A. Deploy to Model Serving

The Databricks SDK `WorkspaceClient` wraps the Model Serving REST API.  
`create_and_wait` blocks until the endpoint reaches the `READY` state.  
Setting `scale_to_zero_enabled=True` ensures the endpoint scales down when idle, saving cost.

In [ ]:
from databricks.sdk import WorkspaceClient
from databricks.sdk.service.serving import (
    EndpointCoreConfigInput,
    ServedEntityInput,
)

w = WorkspaceClient()

served_entity = ServedEntityInput(
    entity_name=MODEL_NAME,
    entity_version=str(MODEL_VERSION),
    workload_size="Small",          # XSmall | Small | Medium | Large
    scale_to_zero_enabled=True,
)

try:
    endpoint = w.serving_endpoints.create_and_wait(
        name=ENDPOINT_NAME,
        config=EndpointCoreConfigInput(served_entities=[served_entity]),
    )
    print(f"Endpoint created: {endpoint.name}  state={endpoint.state.ready}")
except Exception as e:
    if "already exists" in str(e).lower():
        print(f"Endpoint '{ENDPOINT_NAME}' already exists — skipping creation.")
        endpoint = w.serving_endpoints.get(ENDPOINT_NAME)
        print(f"Existing endpoint state: {endpoint.state.ready}")
    else:
        raise

## B. Test the Endpoint

Use `w.serving_endpoints.query` to send a single request using the `dataframe_records` format.  
The response contains `predictions` (or `choices` for Foundation Model API endpoints).

In [ ]:
import json

response = w.serving_endpoints.query(
    name=ENDPOINT_NAME,
    dataframe_records=[{"input": "What is the transformer architecture?"}],
)

# The SDK returns a QueryEndpointResponse object; convert to dict for display
response_dict = response.as_dict()
print(json.dumps(response_dict, indent=2))

## C. A/B Testing

Databricks Model Serving supports **traffic splitting** across multiple served entities within a single endpoint.  
Each `ServedEntityInput` receives a `traffic_percentage`; the values must sum to 100.

Use cases:
- Compare a fine-tuned model against the baseline without changing client code.
- Gradually roll out a new model version (canary release).
- Measure latency or quality differences under production traffic.

Traffic routing is handled transparently by the endpoint — callers always hit the same endpoint URL.

In [ ]:
try:
    from databricks.sdk.service.serving import TrafficConfig, Route

    MODEL_VERSION_2 = 2   # Assumes version 2 has been registered in UC

    served_v1 = ServedEntityInput(
        entity_name=MODEL_NAME,
        entity_version=str(MODEL_VERSION),
        workload_size="Small",
        scale_to_zero_enabled=True,
        name="rag_agent_v1",
    )

    served_v2 = ServedEntityInput(
        entity_name=MODEL_NAME,
        entity_version=str(MODEL_VERSION_2),
        workload_size="Small",
        scale_to_zero_enabled=True,
        name="rag_agent_v2",
    )

    traffic_config = TrafficConfig(
        routes=[
            Route(served_model_name="rag_agent_v1", traffic_percentage=50),
            Route(served_model_name="rag_agent_v2", traffic_percentage=50),
        ]
    )

    updated_endpoint = w.serving_endpoints.update_config_and_wait(
        name=ENDPOINT_NAME,
        served_entities=[served_v1, served_v2],
        traffic_config=traffic_config,
    )

    print(f"A/B config applied. Endpoint state: {updated_endpoint.state.ready}")
    for route in updated_endpoint.config.traffic_config.routes:
        print(f"  {route.served_model_name}: {route.traffic_percentage}%")

except Exception as e:
    print(f"A/B testing requires two registered model versions. Skipping: {e}")
    print("Register a second model version (e.g., from Lab 05) and re-run this cell.")

## D. Batch Inference with `ai_query()`

`ai_query()` is a built-in Databricks SQL function that calls a Model Serving endpoint for every row in a query.  
It enables **batch inference directly in SQL** without writing Python loops or Spark UDFs.

**Syntax:**
```sql
ai_query(
    endpoint_name,   -- string literal: name of the serving endpoint
    request          -- column or expression passed as the model input
)
```

Ideal for scheduled batch jobs, Delta table enrichment, and offline evaluation pipelines.

In [ ]:
# Step 1: Create a small test table with questions
spark.sql(f"""
    CREATE OR REPLACE TABLE {CATALOG}.{SCHEMA}.test_questions (
        id       INT,
        question STRING
    )
""")

spark.sql(f"""
    INSERT INTO {CATALOG}.{SCHEMA}.test_questions VALUES
        (1, 'What is the transformer architecture?'),
        (2, 'Explain attention mechanisms in neural networks.'),
        (3, 'What is retrieval-augmented generation?'),
        (4, 'How does LangChain differ from LlamaIndex?'),
        (5, 'What are the key components of a RAG pipeline?')
""")

# Step 2: Run batch inference using ai_query()
results_df = spark.sql(f"""
    SELECT
        id,
        question,
        ai_query(
            '{ENDPOINT_NAME}',
            question
        ) AS answer
    FROM {CATALOG}.{SCHEMA}.test_questions
    ORDER BY id
""")

display(results_df)

## E. Provisioned Throughput vs Pay-per-Token

Databricks offers two serving tiers for Foundation Models. Choosing the right one depends on your usage pattern.

### Pay-per-Token (Serverless)
- **Billed per token** processed (input + output).
- No upfront commitment; scales automatically to zero when idle.
- Best for: **development, testing, low/bursty traffic** workloads.
- Latency may vary under high concurrency.

### Provisioned Throughput
- **Billed per DBU/hour** regardless of actual token usage.
- Reserved compute: **guaranteed tokens per second (TPS)**, predictable latency.
- Best for: **production workloads with sustained, high-volume traffic**.
- Eliminates cold-start latency; SLA-backed availability.

### Cost Comparison

| Scenario | Pay-per-Token | Provisioned Throughput |
|---|---|---|
| 10 req/day, dev | Low cost | Wasteful (idle hours billed) |
| 1 000 req/min, prod | High variable cost | Lower per-token effective cost |
| Burst traffic | Handled automatically | May exceed provisioned TPS |
| Latency SLA | Not guaranteed | Guaranteed |

### When to Choose
| Choose | When |
|---|---|
| **Pay-per-Token** | Prototyping, internal tools, unpredictable demand |
| **Provisioned Throughput** | Customer-facing APIs, real-time dashboards, batch pipelines with SLA |

> **Exam tip:** The exam tests whether you can identify that `scale_to_zero_enabled` applies to **custom model serving**, not Foundation Model API endpoints, which are always serverless.

## Cleanup — Stop the Billing Clock

The Model Serving endpoint costs money while running. Delete it when done testing.

In [ ]:
# Uncomment to delete the serving endpoint
# w.serving_endpoints.delete(ENDPOINT_NAME)
# print(f"Deleted endpoint: {ENDPOINT_NAME}")

## Key Takeaways

| Concept | Summary |
|---|---|
| **Serving Endpoints** | Deploy UC-registered models via SDK or UI; endpoint URL is stable regardless of model version |
| **Scale-to-Zero** | Enabled per `ServedEntityInput`; reduces cost for low-traffic custom model endpoints |
| **A/B Testing** | Use `TrafficConfig` with multiple `ServedEntityInput` entries summing to 100% traffic |
| **`ai_query()` Batch** | SQL-native batch inference; joins model output directly to table columns |
| **Provisioned vs Serverless** | Choose based on traffic volume, latency SLA, and cost predictability |

## Key Concepts

| Concept | Definition |
|---|---|
| **Model Serving Endpoint** | A managed HTTP endpoint that wraps a registered ML model and handles scaling, routing, and versioning |
| **Scale to Zero** | When enabled, a served entity scales compute to zero replicas during idle periods to minimize cost; `scale_to_zero_enabled=True` in `ServedEntityInput` |
| **ServedEntityInput** | SDK class representing a single model version to be served; configures entity name, version, workload size, and traffic share |
| **A/B Testing (Traffic Split)** | Routing a percentage of incoming requests to different model versions via `TrafficConfig.routes`; percentages must sum to 100 |
| **`ai_query()`** | Built-in Databricks SQL function for calling a Model Serving endpoint row-by-row within a SQL query; enables SQL-native batch inference |
| **Provisioned Throughput** | A serving tier that reserves dedicated compute for Foundation Models; billed per DBU/hour regardless of token usage; guarantees tokens-per-second |
| **Pay-per-Token** | A serverless serving tier billed per input+output token; automatically scales; best for variable or low-volume traffic |

## Exam Preparation

### Exam Questions

**Q1.** You call `w.serving_endpoints.create_and_wait` with `workload_size="Small"` and `scale_to_zero_enabled=True`. After the endpoint is `READY`, what happens if no requests arrive for 30 minutes?

- A) The endpoint is deleted automatically
- B) The served entity scales down to zero replicas, stopping compute billing
- C) The endpoint moves to a `DEGRADED` state
- D) Nothing — `scale_to_zero_enabled` only applies to Foundation Model API endpoints

**Answer: B.** Scale-to-zero reduces replicas to zero during inactivity, stopping compute cost without deleting the endpoint configuration.

---

**Q2.** A data engineer wants to enrich a Delta table with LLM-generated summaries as part of a scheduled SQL job. Which approach is most appropriate?

- A) Write a Python Spark UDF that calls `requests.post` for each row
- B) Use `ai_query()` in a SQL `SELECT` statement referencing the serving endpoint
- C) Export the table to CSV and call the endpoint locally
- D) Use `mlflow.pyfunc.load_model` and apply it with `DataFrame.map`

**Answer: B.** `ai_query()` is the SQL-native function designed for this use case and integrates with scheduled Databricks SQL jobs.

---

**Q3.** You configure an A/B test with three served entities. Entity A gets 40%, Entity B gets 40%, and Entity C gets 30%. What happens when you call `update_config_and_wait`?

- A) The call succeeds and excess traffic is dropped
- B) Traffic is normalized automatically to 100%
- C) The API returns a validation error because percentages sum to 110%
- D) Entity C receives 0% traffic as the excess is redistributed

**Answer: C.** Databricks validates that traffic percentages sum to exactly 100 and returns an error if they do not.

---

**Q4.** You use `ai_query('my-endpoint', input_col)` in a SQL query. The endpoint is currently in `UPDATING` state. What is the expected behavior?

- A) The query waits indefinitely until the endpoint becomes `READY`
- B) The query returns `NULL` for all rows
- C) The query fails with an error because the endpoint is not available
- D) The query routes requests to the previous configuration during the update

**Answer: C.** `ai_query()` requires the endpoint to be in `READY` state; requests to an `UPDATING` endpoint fail.

---

**Q5.** Your production chatbot handles 800 requests per minute consistently throughout business hours. You are evaluating Provisioned Throughput vs Pay-per-Token. Which statement best describes the cost trade-off?

- A) Pay-per-Token is always cheaper because you only pay for tokens used
- B) Provisioned Throughput has a higher per-DBU cost but lower effective per-token cost at sustained high volume
- C) The two options have identical cost at any volume
- D) Provisioned Throughput is only available for custom models, not Foundation Models

**Answer: B.** At sustained high request rates, the fixed DBU/hour cost of Provisioned Throughput amortizes to a lower effective per-token cost than pay-per-token billing.

## Cost Breakdown

| Resource | Estimated Cost |
|---|---|
| Databricks serverless compute (notebook runtime, ~45 min) | ~$0.50 |
| LLM token usage (test queries via endpoint) | ~$1.00 |
| Model Serving endpoint runtime (~45 min, Small workload size) | ~$2.00 |
| **Total** | **~$3.50 – $5.00** |

**Estimated time:** 45 min | **Estimated cost:** $3 – $5

> Costs vary by region and DBU pricing tier. Use `scale_to_zero_enabled=True` and delete the endpoint after the lab to avoid ongoing charges.